# ARC-AGI-3 Kaggle Submission Notebook

Notebook starter para ARC-AGI-3 con agente heurístico + memoria + feedback por reward.

Este notebook está pensado como base técnica. El bloque final de export puede necesitar adaptación al formato exacto de submission exigido por la competición.

## 1. Instalación de dependencias
Se fijan versiones mínimas para reducir conflictos frecuentes del entorno de Kaggle sin forzar una versión de gym.

In [ ]:
!pip install -q "arc-agi" "pillow<12"

## 2. Imports

In [ ]:
import json
import os
import random
from collections import defaultdict

import arc_agi


## 3. Configuración
Ajustá estos parámetros según el juego o el volumen de pruebas que quieras ejecutar.

In [ ]:
MAX_STEPS = 100
GAME_ID = "ls20"
EXPORT_PATH = "/kaggle/working/submission.json"

## 4. Definición del agente
Agente simple con:
- memoria de estados visitados
- penalización de acciones repetidas
- refuerzo básico según reward
- coordenadas exploratorias para acciones complejas

In [ ]:
class SubmissionAgent:
    def __init__(self, max_steps=100):
        self.max_steps = max_steps
        self.coord_candidates = [
            (0, 0), (0, 15), (15, 0), (15, 15),
            (3, 3), (7, 7), (8, 4), (4, 8), (10, 10)
        ]
        self.reset()

    def reset(self):
        self.step_count = 0
        self.visited_states = set()
        self.action_counts = defaultdict(int)
        self.action_success = defaultdict(float)
        self.last_reward = 0.0

    def state_key(self, obs):
        return repr(obs)[:1000]

    def act(self, env, obs):
        if obs is not None:
            self.visited_states.add(self.state_key(obs))

        actions = list(env.action_space)

        def action_score(action):
            name = str(action)
            success_bonus = self.action_success[name]
            usage_penalty = self.action_counts[name]
            return success_bonus - 0.25 * usage_penalty

        actions.sort(key=action_score, reverse=True)
        pool = actions[:max(1, len(actions) // 2)]
        action = random.choice(pool)

        action_data = {}
        if hasattr(action, "is_complex") and action.is_complex():
            x, y = random.choice(self.coord_candidates)
            action_data = {"x": x, "y": y}

        self.action_counts[str(action)] += 1
        return action, action_data

    def update(self, action, reward):
        delta = reward - self.last_reward
        self.action_success[str(action)] += delta
        self.last_reward = reward


## 5. Utilidad para normalizar la salida de env.step
Algunos entornos devuelven 5 valores, otros más. Esta función los normaliza.

In [ ]:
def parse_step_result(step_result):
    if not isinstance(step_result, tuple):
        return step_result, 0.0, False, False, {}

    if len(step_result) >= 5:
        obs = step_result[0]
        reward = step_result[1]
        terminated = step_result[2]
        truncated = step_result[3]
        info = step_result[4] if isinstance(step_result[4], dict) else {}
        return obs, reward, terminated, truncated, info

    if len(step_result) == 4:
        obs, reward, done, info = step_result
        return obs, reward, bool(done), False, info if isinstance(info, dict) else {}

    if len(step_result) == 3:
        obs, reward, done = step_result
        return obs, reward, bool(done), False, {}

    if len(step_result) == 2:
        obs, reward = step_result
        return obs, reward, False, False, {}

    return step_result[0], 0.0, False, False, {}


## 6. Runner
Ejecuta un único juego y devuelve trace + scorecard.

In [ ]:
def run_single_game(game_id, max_steps=100):
    arc = arc_agi.Arcade()
    env = arc.make(game_id, render_mode=None)
    agent = SubmissionAgent(max_steps=max_steps)

    obs = None
    reward = 0.0
    terminated = False
    truncated = False
    info = {}
    trace = []

    for step in range(agent.max_steps):
        action, action_data = agent.act(env, obs)
        step_result = env.step(action, data=action_data)
        obs, reward, terminated, truncated, info = parse_step_result(step_result)
        agent.update(action, reward)

        trace.append({
            "step": step,
            "action": str(action),
            "action_data": action_data,
            "reward": reward,
            "terminated": terminated,
            "truncated": truncated,
            "info": info,
            "raw_step_result_len": len(step_result) if isinstance(step_result, tuple) else None
        })

        if terminated or truncated:
            break

    return {
        "game_id": game_id,
        "scorecard": str(arc.get_scorecard()),
        "trace": trace
    }


## 7. Ejecución

In [ ]:
result = run_single_game(GAME_ID, max_steps=MAX_STEPS)
result

## 8. Export de submission
Este bloque exporta un JSON base.

Si la competición exige otro esquema, adaptá únicamente esta celda final para respetar el formato oficial.

In [ ]:
os.makedirs("/kaggle/working", exist_ok=True)

with open(EXPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("Archivo exportado:", EXPORT_PATH)

## 9. Nota final
Antes de hacer submit, compará este JSON contra el sample submission oficial de la competencia.